In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py
import numpy as np

hdf5_path = '../data/EPIC_audio.hdf5' 


print(f"Tentativo di apertura del file: {hdf5_path}...\n")

with h5py.File(hdf5_path, 'r') as f:
    
    keys = list(f.keys())
    print(f"File aperto con successo.")
    print(f"Numero totale di chiavi (elementi) nel file: {len(keys)}")
    
    print(f"Prime 5 chiavi: {keys[:5]}\n")
    
    if len(keys) > 0:
        first_key = keys[0]
        audio_data = f[first_key]
        
        print(f"--- Analisi del primo elemento: '{first_key}' ---")
        print(f"Classe dell'oggetto HDF5: {type(audio_data)}")
        
        print(f"Forma (Shape) dei dati: {audio_data.shape}") 
        
        print(f"Tipo di dato (dtype): {audio_data.dtype}\n")
        
        primi_10_campioni = audio_data[:10]
        print(f"Primi 10 valori grezzi dell'audio:\n{primi_10_campioni}")

Tentativo di apertura del file: ../data/EPIC_audio.hdf5...

File aperto con successo.
Numero totale di chiavi (elementi) nel file: 712
Prime 5 chiavi: ['P01_01', 'P01_02', 'P01_03', 'P01_04', 'P01_05']

--- Analisi del primo elemento: 'P01_01' ---
Classe dell'oggetto HDF5: <class 'h5py._hl.dataset.Dataset'>
Forma (Shape) dei dati: (39651328,)
Tipo di dato (dtype): float32

Primi 10 valori grezzi dell'audio:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [3]:
import sys

sys.path.append('../src') 
from datasets.epic_sounds import EPICSoundsDataset

dataset = EPICSoundsDataset(
    annotations_file='../data/epic-sounds-annotations/EPIC_Sounds_train.csv',
    hdf5_path='../data/EPIC_audio.hdf5'
)

print(f"Numero totale di clip audio pronte: {len(dataset)}")

spettrogramma, etichetta = dataset[0]

print(f"Etichetta (Classe ID): {etichetta}")
print(f"Shape dell'immagine dello Spettrogramma: {spettrogramma.shape}")

Numero totale di clip audio pronte: 60055
Etichetta (Classe ID): 6
Shape dell'immagine dello Spettrogramma: torch.Size([1, 128, 1024])


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

batch_spectrograms, batch_labels = next(iter(train_loader))

print(f"Shape del batch di spettrogrammi: {batch_spectrograms.shape}")
print(f"Shape del batch di etichette: {batch_labels.shape}")

Shape del batch di spettrogrammi: torch.Size([32, 1, 128, 1024])
Shape del batch di etichette: torch.Size([32])


In [ ]:
import sys
sys.path.append('../src')
import torch
from datasets.epic_sounds import EPICSoundsDataset
from torch.utils.data import DataLoader
from models.ast import EPICASTBaseline


dataset = EPICSoundsDataset(
    annotations_file='../data/epic-sounds-annotations/EPIC_Sounds_train.csv',
    hdf5_path='../data/EPIC_audio.hdf5'
)

train_loader = DataLoader(dataset, batch_size=4, shuffle=True)
real_input_batch, labels = next(iter(train_loader))

print(f"Shape del batch reale: {real_input_batch.shape}") 

print("Inizializzazione AST...")
model = EPICASTBaseline(num_classes=44)

print("Esecuzione del forward pass...")
logits = model(real_input_batch)

print(f"Shape dei logits di output: {logits.shape}")

Shape del batch reale: torch.Size([4, 1, 128, 1024])
Inizializzazione AST...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Esecuzione del forward pass...
Shape dei logits di output: torch.Size([4, 44])


In [ ]:
import pandas as pd


df_train = pd.read_csv('../data/epic-sounds-annotations/EPIC_Sounds_train.csv')


NUM_CLASSES = df_train['class_id'].nunique()
print(f"Numero esatto di classi nel dataset EPIC-Sounds: {NUM_CLASSES}")

Numero esatto di classi nel dataset EPIC-Sounds: 44


In [ ]:
import torch.nn as nn
import torch.optim as optim

print(f"Inizializzazione AST con {NUM_CLASSES} classi...")
model = EPICASTBaseline(num_classes=NUM_CLASSES)


device = torch.device("cpu")
print(f"Utilizzo del dispositivo hardware: {device}")
model.to(device)


criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=1e-4)

model.train()

print("\n--- INIZIO TRAINING LOOP (Test su 2 Batch) ---")


for batch_idx, (inputs, labels) in enumerate(train_loader):
    

    inputs = inputs.to(device)
    labels = labels.to(device)
    

    optimizer.zero_grad()
    
    outputs = model(inputs)
    
    loss = criterion(outputs, labels)
    
    loss.backward()
    
    optimizer.step()
    
    print(f"Batch [{batch_idx+1}] completato | Loss scalare: {loss.item():.4f}")
    
    if batch_idx == 1:
        print("Test del Training Loop concluso con successo. Nessun errore architetturale rilevato.")
        break

Inizializzazione AST con 44 classi...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Utilizzo del dispositivo hardware: cpu

--- INIZIO TRAINING LOOP (Test su 2 Batch) ---
Batch [1] completato | Loss scalare: 3.7207
Batch [2] completato | Loss scalare: 3.6515
Test del Training Loop concluso con successo. Nessun errore architetturale rilevato.
